# OpenAlex versión de buscado por 1 autor por Nombre

In [ ]:
import requests
import pandas as pd
import time


AUTHOR_NAME = "Daniela Quiñones"  #  cambiar por el nombre exacto
CONTACT_EMAIL = "gerard.otto.s@mail.pucv.cl"         #  email académico
SLEEP_BETWEEN = 0.3                            # segundos entre requests

headers = {
    "User-Agent": f"MyResearchBot/1.0 (mailto:{CONTACT_EMAIL})"
}
BASE_URL = "https://api.openalex.org"

# =============================================
# Buscar autor por nombre
# =============================================
print(f"🔍 Buscando autor '{AUTHOR_NAME}' en OpenAlex...")

resp = requests.get(
    f"{BASE_URL}/authors",
    params={
        "search": AUTHOR_NAME,
        "per-page": 5,
        "mailto": CONTACT_EMAIL
    },
    headers=headers
)
resp.raise_for_status()
data = resp.json()

if len(data.get("results", [])) == 0:
    raise SystemExit("❌ No se encontraron autores con ese nombre.")

print(f"✅ Se encontraron {len(data['results'])} posibles coincidencias:\n")
for i, a in enumerate(data["results"], 1):
    aff = a.get("last_known_institution", {}).get("display_name", "Sin institución")
    print(f"{i}. {a['display_name']} — {aff} — {a['id']}")

# =============================================
#  Seleccionar autor correcto (por índice)
# =============================================
idx = int(input("\n👉 Ingresa el número del autor correcto: ")) - 1
author = data["results"][idx]
author_id_openalex = author["id"]

print(f"\n✅ Autor seleccionado: {author['display_name']}")
print(f"🆔 OpenAlex ID: {author_id_openalex}")
print(f"🏢 Afiliación: {author.get('last_known_institution', {}).get('display_name', 'No disponible')}")

# =============================================
#  Descargar todas las obras del autor
# =============================================
print("\n📚 Descargando publicaciones...")

works = []
page = 1
while True:
    url = f"{BASE_URL}/works"
    params = {
        "filter": f"authorships.author.id:{author_id_openalex}",
        "per-page": 200,
        "page": page,
        "mailto": CONTACT_EMAIL
    }
    r = requests.get(url, params=params, headers=headers)
    if r.status_code != 200:
        print(f"⚠️ Error {r.status_code} en página {page}.")
        break
    results = r.json().get("results", [])
    if not results:
        break
    works.extend(results)
    print(f"  🧾 Página {page}: {len(results)} artículos.")
    time.sleep(SLEEP_BETWEEN)
    page += 1

print(f"\n✅ Total de publicaciones obtenidas: {len(works)}")

# =============================================
#  Extraer coautores y metadatos (versión robusta)
# =============================================
records = []
for w in works:
    title = w.get("title", "")
    doi = w.get("doi", "")
    pub_year = w.get("publication_year", "")

    # Manejo seguro de 'primary_location' que puede venir como None
    primary_loc = w.get("primary_location") or {}
    source = primary_loc.get("source") or {}
    pub_venue = source.get("display_name", "Desconocido")

    cited_by = w.get("cited_by_count", 0)

    # Coautores
    coauthors = [auth["author"]["display_name"] for auth in w.get("authorships", []) if auth.get("author")]

    records.append({
        "titulo": title,
        "año": pub_year,
        "revista": pub_venue,
        "doi": doi,
        "citaciones": cited_by,
        "n_coautores": len(coauthors),
        "coautores": ", ".join(coauthors)
    })

# =============================================
#  Guardar resultados
# =============================================
df = pd.DataFrame(records)
df = df.sort_values(by="año", ascending=False)
df.to_csv("openalex_autor_publicaciones.csv", index=False)

print("\n💾 Archivo 'openalex_autor_publicaciones.csv' guardado correctamente.")
print(f"🧮 Total de artículos: {len(df)}")
display(df.head(10))



🔍 Buscando autor 'Daniela Quiñones' en OpenAlex...
✅ Se encontraron 5 posibles coincidencias:

1. Daniela Quiñones — Sin institución — https://openalex.org/A5001623713
2. Diego H. Quiñones — Sin institución — https://openalex.org/A5078340594
3. P. Gómez — Sin institución — https://openalex.org/A5050780785
4. María Daniela Mares-Quiñones — Sin institución — https://openalex.org/A5050122940
5. Nadia Daniela Quiñones — Sin institución — https://openalex.org/A5090317561

👉 Ingresa el número del autor correcto: 1

✅ Autor seleccionado: Daniela Quiñones
🆔 OpenAlex ID: https://openalex.org/A5001623713
🏢 Afiliación: No disponible

📚 Descargando publicaciones...
  🧾 Página 1: 61 artículos.

✅ Total de publicaciones obtenidas: 61

💾 Archivo 'openalex_autor_publicaciones.csv' guardado correctamente.
🧮 Total de artículos: 61


,titulo,año,revista,doi,citaciones,n_coautores,coautores
59,Strategies for developing user experience eval...,2025,Procedia Computer Science,https://doi.org/10.1016/j.procs.2025.07.091,0,3,"José Osega, Daniela Quiñones, Luis Rojas"
60,HEUXIVA: A Set of Heuristics for Evaluating Us...,2025,Applied Sciences,https://doi.org/10.3390/app152011178,0,6,"Daniela Quiñones, Luis Rojas, Camila Serrá, Ju..."
47,Exploring practitioners’ perspective on user e...,2025,Procedia Computer Science,https://doi.org/10.1016/j.procs.2025.07.048,1,4,"Luis Rojas, Daniela Quiñones, Claudio Cubillos..."
28,User experience heuristics for in-vehicle info...,2024,Procedia Computer Science,https://doi.org/10.1016/j.procs.2024.05.159,2,3,"Daniela Quiñones, Luis Rojas, Andrés Barraza"
13,UXH-GEDAPP: A set of user experience heuristic...,2024,Information and Software Technology,https://doi.org/10.1016/j.infsof.2024.107408,3,4,"Daniela Quiñones, Claudia Ojeda, Rodrigo F. He..."
14,Innovating Statistics Education: The Design of...,2024,Applied Sciences,https://doi.org/10.3390/app14188515,3,6,"Daniela Quiñones, Felipe Ruz, Jaime Díaz, Fred..."
15,FRAMUX-EV: A Framework for Evaluating User Exp...,2024,Applied Sciences,https://doi.org/10.3390/app14198991,3,3,"Luis Rojas, Daniela Quiñones, Claudio Cubillos"
56,"Peer Review #1 of ""Understanding the customer ...",2023,Desconocido,https://doi.org/10.7287/peerj-cs.1219v0.2/revi...,0,2,"Daniela Quiñones, Luis Rojas"
57,"Peer Review #1 of ""Understanding the customer ...",2023,Desconocido,https://doi.org/10.7287/peerj-cs.1219v0.1/revi...,0,2,"Daniela Quiñones, Luis Rojas"
58,"Peer Review #2 of ""Understanding the customer ...",2023,Desconocido,https://doi.org/10.7287/peerj-cs.1219v0.1/revi...,0,2,"Daniela Quiñones, Luis Rojas"


# **OpenAlex Upgrade (Búsqueda múltiples autores por institución)**

In [ ]:
# ===========================================================
# 🔥 Descarga TOTAL de publicaciones de la PUCV desde OpenAlex
# ===========================================================

import requests
import pandas as pd
import os
import time
from tqdm.auto import tqdm

INSTITUTION_NAME = "Pontificia Universidad Católica de Valparaíso"
CONTACT_EMAIL = "gerard.otto.s@mail.pucv.cl"

BASE_URL = "https://api.openalex.org"
HEADERS = {
    "User-Agent": f"MyResearchBot/1.0 (mailto:{CONTACT_EMAIL})"
}

# ===========================================================
# 1️⃣ Buscar institución
# ===========================================================
print(f"Buscando institución '{INSTITUTION_NAME}' en OpenAlex...")

resp = requests.get(
    f"{BASE_URL}/institutions",
    params={"search": INSTITUTION_NAME, "mailto": CONTACT_EMAIL},
    headers=HEADERS
)
resp.raise_for_status()
data = resp.json()

if not data["results"]:
    raise SystemExit("ERROR: No se encontró la institución.")

inst = data["results"][0]
inst_id = inst["id"].split("/")[-1]

print(f"Institución encontrada: {inst['display_name']}")
print(f"OpenAlex ID: {inst_id}")


# ===========================================================
# 2️⃣ Determinar el rango de años realmente existente
# ===========================================================
print("\nCalculando rango de años disponibles...")

# Consulta mínima para obtener fecha de la publicación más antigua
test = requests.get(
    f"{BASE_URL}/works",
    params={
        "filter": f"institutions.id:{inst_id}",
        "sort": "publication_year:asc",
        "per-page": 1,
        "mailto": CONTACT_EMAIL
    },
    headers=HEADERS
).json()

first_year = test["results"][0]["publication_year"]

# Consulta máxima para obtener la publicación más reciente
test2 = requests.get(
    f"{BASE_URL}/works",
    params={
        "filter": f"institutions.id:{inst_id}",
        "sort": "publication_year:desc",
        "per-page": 1,
        "mailto": CONTACT_EMAIL
    },
    headers=HEADERS
).json()

last_year = test2["results"][0]["publication_year"]

print(f"Años disponibles: {first_year} – {last_year}")


# ===========================================================
# 3️⃣ Función para descargar todos los resultados de un año
# ===========================================================

def download_year(year, inst_id):
    """Descarga todos los works de un año específico."""

    all_works = []
    page = 1

    while True:
        params = {
            "filter": f"institutions.id:{inst_id},publication_year:{year}",
            "per-page": 200,
            "page": page,
            "mailto": CONTACT_EMAIL
        }

        r = requests.get(f"{BASE_URL}/works", params=params, headers=HEADERS)

        # Manejo de bloqueos
        if r.status_code == 403:
            print(f"403 (bloqueo). Esperando 20s...")
            time.sleep(20)
            continue
        if r.status_code == 429:
            print("429 Too Many Requests. Esperando 15s...")
            time.sleep(15)
            continue

        if r.status_code == 400:
            # no es error real; significa que ya no hay más páginas
            break

        if r.status_code != 200:
            print(f"Error {r.status_code} en año {year}, página {page}.")
            break

        results = r.json().get("results", [])
        if not results:
            break

        all_works.extend(results)
        page += 1

        time.sleep(1.3)  # Evita gatillar rate limits

    return all_works


# ===========================================================
# 4️⃣ Descargar año por año (con caché incremental)
# ===========================================================

os.makedirs("openalex_cache_pucv", exist_ok=True)

all_records = []

print("\nDescargando publicaciones por año:")
for year in tqdm(range(first_year, last_year + 1)):

    cache_file = f"openalex_cache_pucv/pucv_{year}.json"

    # Si ya existe el archivo, lo cargamos
    if os.path.exists(cache_file):
        works_year = pd.read_json(cache_file, orient="records").to_dict("records")
    else:
        works_year = download_year(year, inst_id)
        pd.DataFrame(works_year).to_json(cache_file, orient="records", indent=2)

    all_records.extend(works_year)


print(f"\nTotal bruto de publicaciones recopiladas: {len(all_records)}")


# ===========================================================
# 5️⃣ Depuración y extracción de coautores
# ===========================================================

clean_rows = []

for w in all_records:
    title = w.get("title") or ""
    doi = w.get("doi") or ""
    year = w.get("publication_year") or None
    cited_by = w.get("cited_by_count", 0)

    primary_loc = w.get("primary_location") or {}
    journal = (primary_loc.get("source") or {}).get("display_name") or "Desconocido"

    authorships = w.get("authorships", [])

    # Lista limpia de autores
    author_names = [
        a["author"]["display_name"]
        for a in authorships
        if a.get("author") and isinstance(a["author"].get("display_name"), str)
    ]

    for a in authorships:
        if not a.get("author"):
            continue

        author_name = a["author"].get("display_name")
        if not isinstance(author_name, str):
            continue

        coauthors = [n for n in author_names if n != author_name]

        clean_rows.append({
            "autor": author_name,
            "titulo": title,
            "año": year,
            "revista": journal,
            "doi": doi,
            "citaciones": cited_by,
            "n_coautores": len(coauthors),
            "coautores": ", ".join(coauthors)
        })


# ===========================================================
# 6️⃣ Guardar resultados finales
# ===========================================================

df = pd.DataFrame(clean_rows)

# Eliminar duplicados por autor + título
df = df.drop_duplicates(subset=["autor", "titulo"])

df.to_csv("openalex_pucv_publicaciones_REMAKE.csv", index=False)

print("\n=====================================")
print("Descarga COMPLETA finalizada.")
print("Archivo generado: openalex_pucv_publicaciones_TODAS.csv")
print(f"Autores únicos: {df['autor'].nunique()}")
print(f"Publicaciones únicas: {df['titulo'].nunique()}")
print(f"Registros finales (autor-publicación): {len(df)}")
print("=====================================")

df.head(10)


Buscando institución 'Pontificia Universidad Católica de Valparaíso' en OpenAlex...
Institución encontrada: Pontificia Universidad Católica de Valparaíso
OpenAlex ID: I130474213

Calculando rango de años disponibles...
Años disponibles: 1946 – 2025

Descargando publicaciones por año:


  0%|          | 0/80 [00:00<?, ?it/s]


Total bruto de publicaciones recopiladas: 16470

Descarga COMPLETA finalizada.
Archivo generado: openalex_pucv_publicaciones_TODAS.csv
Autores únicos: 28273
Publicaciones únicas: 16034
Registros finales (autor-publicación): 75834


Loading ITables v2.6.2 from the internet... (need help?)


# **Formatear tablas Buscador True/False**

In [ ]:
!pip install itables --quiet
import itables
itables.init_notebook_mode(all_interactive=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.4 MB/s eta 0:00:00


In [ ]:
display(tabla)

Output hidden; open in https://colab.research.google.com to view.

# Reformateo visual de los datos (mejorar legibilidad, skipeo de lineas rotas)

In [ ]:
import pandas as pd

ruta = "/content/openalex_pucv_publicaciones_TODAS.csv"

# Intento robusto para leer CSV con errores de formato
try:
    df = pd.read_csv(
        ruta,
        on_bad_lines="skip",  # omite líneas rotas
        engine="python",      # más flexible que el parser por defecto
        quotechar='"',
        escapechar='\\'
    )
    print(f"✅ CSV cargado con {len(df)} filas y {len(df.columns)} columnas.")
except Exception as e:
    print(f"❌ Error al leer el CSV: {e}")

df


✅ CSV cargado con 75521 filas y 8 columnas.


Loading ITables v2.5.2 from the internet... (need help?)


# **Análisis bibliométrico completo**

In [ ]:
import pandas as pd

#Cargar dataset
df = pd.read_csv("openalex_pucv_publicaciones_TODAS.csv")
df['año'] = pd.to_numeric(df['año'], errors='coerce')
df['citaciones'] = pd.to_numeric(df['citaciones'], errors='coerce').fillna(0)

#Producción científica anual
produccion_anual = df.groupby("año")["titulo"].nunique()
display(produccion_anual.tail(15))

#Citaciones totales por año
citaciones_anuales = df.groupby("año")["citaciones"].sum()
display(citaciones_anuales.tail(15))

#Autores más productivos (Top 50)
top_autores = df.groupby("autor")["titulo"].nunique().sort_values(ascending=False)
display(top_autores)

#Autores más citados (Top 50)
top_citados = df.groupby("autor")["citaciones"].sum().sort_values(ascending=False)
display(top_citados)

#Revistas donde publica más la PUCV
top_revistas = df.groupby("revista")["titulo"].nunique().sort_values(ascending=False)
display(top_revistas)


,titulo
año,
2011,450
2012,434
2013,510
2014,639
2015,715
2016,910
2017,873
2018,994
2019,1079


,citaciones
año,
2011,38675
2012,35098
2013,36212
2014,94548
2015,55355
2016,84404
2017,71432
2018,96820
2019,223073


,titulo
autor,
Broderick Crawford,357
Ricardo Soto,326
Fanny Guzmán,199
Víctor Leiva,192
Andrés Illanes,165
...,...
Dos Santos,0
Daniel Duclos,0
Christopher Morrison,0


,citaciones
autor,
Emmanuel N. Saridakis,5951
Andrés Illanes,5557
Ali Övgün,3882
Rolando Chamy,3568
Víctor Leiva,3372
...,...
Óscar Andrade Castro,0
Íñigo Andrés de la Maza-Gazmuri,0
Ignacio Salinas Falcone,0


,titulo
revista,
Desconocido,1188
Lecture notes in computer science,194
Revista signos,156
Revista de estudios histórico-jurídicos,150
DOAJ (DOAJ: Directory of Open Access Journals),146
...,...
telondefondo Revista de Teoría y Crítica Teatral,1
eScholarship (California Digital Library),1
iQual Revista de Género e Igualdad,1


# **Redes de Coautoría** (para Gephi y NetworkX)

La red de coautoría es una gráfica autor–autor donde:

- Nodo: un autor
- Arista: colaboración en al menos un artículo
- Peso de la arista: número de publicaciones en conjunto


In [ ]:
# Construir red de coautoría robusta y exportar a GEXF (Gephi)
from itertools import combinations
import networkx as nx
import pandas as pd

# --- Asegurar que columnas existen ---
if "autor" not in df.columns or "coautores" not in df.columns:
    raise RuntimeError("El DataFrame debe contener columnas 'autor' y 'coautores'")

# --- Normalizar y rellenar NaN ---
df = df.copy()
df["autor"] = df["autor"].astype(str).replace({"nan": ""}).str.strip()
# Si coautores es NaN, reemplazar por cadena vacía
df["coautores"] = df["coautores"].fillna("").astype(str)

G = nx.Graph()

# --- Construcción de aristas ---
for idx, row in df.iterrows():
    autor_principal = row.get("autor", "")
    if not isinstance(autor_principal, str) or autor_principal.strip() == "":
        # Si autor principal inválido, saltar fila
        continue
    autor_principal = autor_principal.strip()

    # Parsear lista de coautores; tolerar separadores extra y espacios
    raw = row.get("coautores", "")
    # Si coautores viene como lista en vez de string, manejarlo
    if isinstance(raw, (list, tuple)):
        coautores_list = [str(x).strip() for x in raw if str(x).strip()]
    else:
        # separar por coma; se puede extender a otros separadores si es necesario
        coautores_list = [a.strip() for a in raw.split(",") if a and a.strip()]

    # Construir conjunto único de autores participando en el registro
    autores_en_articulo = {autor_principal}
    for c in coautores_list:
        if isinstance(c, str) and c:
            autores_en_articulo.add(c)

    # Si solo hay un autor en el artículo, no se crean aristas (opcional: añadir nodo)
    if len(autores_en_articulo) == 1:
        # Asegurar que el nodo exista (para calcular degree later)
        node = next(iter(autores_en_articulo))
        if not G.has_node(node):
            G.add_node(node)
        continue

    # Crear/actualizar aristas entre todas las combinaciones de autores
    for a1, a2 in combinations(sorted(autores_en_articulo), 2):
        if a1 == a2:
            continue
        if G.has_edge(a1, a2):
            G[a1][a2]["weight"] += 1
            G[a1][a2]["count"] += 1  # count redundante si lo prefieres
        else:
            G.add_edge(a1, a2, weight=1, count=1)

# --- Añadir métricas de nodo: degree, strength (suma de pesos) ---
# degree (número de vecinos), strength (suma de pesos de aristas)
for node in G.nodes():
    deg = G.degree(node)
    # fuerza / strength: suma de pesos de aristas incidentes
    strength = sum(d.get("weight", 1) for _, _, d in G.edges(node, data=True))
    G.nodes[node]["degree"] = deg
    G.nodes[node]["strength"] = strength

# --- Exportar red a GEXF para Gephi ---
output_gexf = "red_coautoria_pucv.gexf"
nx.write_gexf(G, output_gexf)
print(f"Red exportada: {output_gexf}")
print(f"Nodos: {G.number_of_nodes()}, Aristas: {G.number_of_edges()}")


Red exportada: red_coautoria_pucv.gexf
Nodos: 28249, Aristas: 293062


Este archivo se puede abrir directamente en Gephi.

Recomendado:

- Layout: ForceAtlas2

- Color por degree o betweenness centrality

- Tamaño por número de colaboraciones acumuladas (node strength)

# **Métricas bibliométricas individuales**
(h-index, g-index, PageRank científico)

| Métrica          | Descripción                            |
| ---------------- | -------------------------------------- |
| publicaciones    | Número de artículos únicos             |
| total_citaciones | Citaciones acumuladas                  |
| cit_promedio     | Citaciones promedio                    |
| max_citaciones   | Publ. con más citaciones               |
| h_index          | Número h                               |
| g_index          | Número g (sensible a citaciones altas) |


In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Normalización de datos previos
# ---------------------------------------------------------

# Asegurar que citaciones sea numérico
df["citaciones"] = pd.to_numeric(df["citaciones"], errors="coerce").fillna(0)

# Asegurar que año sea numérico
df["año"] = pd.to_numeric(df["año"], errors="coerce")

# Normalizar autor (string)
df["autor"] = df["autor"].fillna("Desconocido").astype(str)

# ---------------------------------------------------------
# Funciones de métricas bibliométricas
# ---------------------------------------------------------

def h_index(citations):
    """
    Calcula el h-index.
    """
    citations_sorted = sorted(citations, reverse=True)
    return sum(c >= i+1 for i, c in enumerate(citations_sorted))


def g_index(citations):
    """
    Calcula el g-index.
    """
    citations_sorted = sorted(citations, reverse=True)
    cumulative = 0
    g = 0
    for i, c in enumerate(citations_sorted, start=1):
        cumulative += c
        if cumulative >= i*i:
            g = i
    return g

# ---------------------------------------------------------
# Cálculo de métricas por autor
# ---------------------------------------------------------

metricas = (
    df.groupby("autor")
      .agg(
          publicaciones=("titulo", "nunique"),
          total_citaciones=("citaciones", "sum"),
          cit_promedio=("citaciones", "mean"),
          max_citaciones=("citaciones", "max"),
      )
      .reset_index()
)

# Obtener listas de citaciones de cada autor (para h-index y g-index)
citaciones_por_autor = df.groupby("autor")["citaciones"].apply(list)

# Calcular h-index y g-index agregados
metricas["h_index"] = metricas["autor"].apply(lambda a: h_index(citaciones_por_autor.get(a, [])))
metricas["g_index"] = metricas["autor"].apply(lambda a: g_index(citaciones_por_autor.get(a, [])))

# Redondeo de promedios
metricas["cit_promedio"] = metricas["cit_promedio"].round(2)

# Ordenar por autores más influyentes (citaciones totales)
metricas = metricas.sort_values(by="total_citaciones", ascending=False)

# ---------------------------------------------------------
# Exportar resultados
# ---------------------------------------------------------

metricas.to_csv("metricas_bibliometricas_pucv.csv", index=False)

print("Archivo 'metricas_bibliometricas_pucv.csv' generado correctamente.")
display(metricas)


Archivo 'metricas_bibliometricas_pucv.csv' generado correctamente.


Loading ITables v2.6.1 from the internet... (need help?)


## **Simulador Top 2% Scientist**

In [ ]:
import pandas as pd
from collections import defaultdict

# Cargar dataset
df = pd.read_csv("/content/openalex_pucv_publicaciones_REMAKE.csv")

# Diccionario acumulador
conteo_autores = defaultdict(lambda: {
    "primer_autor": 0,
    "ultimo_autor": 0,
    "coautor_intermedio": 0,
    "citas_primer_autor": 0,
    "citas_ultimo_autor": 0,
    "citas_coautor_intermedio": 0,
    "citas_totales": 0
})

# Procesar paper por paper
for doi, grupo in df.groupby("doi"):

    fila = grupo.iloc[0]

    # Total de citas del paper
    citas_paper = fila["citaciones"]

    # Construir lista ordenada de autores
    autores = [fila["autor"]]

    if pd.notna(fila["coautores"]):
        coautores = [a.strip() for a in fila["coautores"].split(",")]
        autores.extend(coautores)

    # Caso autor único
    if len(autores) == 1:
        autor = autores[0]
        conteo_autores[autor]["primer_autor"] += 1
        conteo_autores[autor]["ultimo_autor"] += 1
        conteo_autores[autor]["citas_primer_autor"] += citas_paper
        conteo_autores[autor]["citas_ultimo_autor"] += citas_paper
        conteo_autores[autor]["citas_totales"] += citas_paper

    else:
        # Primer autor
        primer = autores[0]
        conteo_autores[primer]["primer_autor"] += 1
        conteo_autores[primer]["citas_primer_autor"] += citas_paper
        conteo_autores[primer]["citas_totales"] += citas_paper

        # Último autor
        ultimo = autores[-1]
        conteo_autores[ultimo]["ultimo_autor"] += 1
        conteo_autores[ultimo]["citas_ultimo_autor"] += citas_paper
        conteo_autores[ultimo]["citas_totales"] += citas_paper

        # Coautores intermedios
        for autor in autores[1:-1]:
            conteo_autores[autor]["coautor_intermedio"] += 1
            conteo_autores[autor]["citas_coautor_intermedio"] += citas_paper
            conteo_autores[autor]["citas_totales"] += citas_paper

# Convertir a DataFrame
resultado = (
    pd.DataFrame.from_dict(conteo_autores, orient="index")
      .reset_index()
      .rename(columns={"index": "autor"})
)

# Ordenar por impacto total
resultado = resultado.sort_values(
    by="citas_totales",
    ascending=False
)

resultado.head()


Loading ITables v2.6.2 from the internet... (need help?)


In [ ]:
display(resultado)

Loading ITables v2.6.2 from the internet... (need help?)


## **Uso del ranking 2%**

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

# =====================================================
# 1. Cargar dataset base OpenAlex
# =====================================================
df_base = pd.read_csv("/content/openalex_pucv_publicaciones_REMAKE.csv")

# -----------------------------------------------------
# Calcular número de autores por paper
# -----------------------------------------------------
def contar_autores(fila):
    if pd.isna(fila["coautores"]):
        return 1
    return 1 + len(fila["coautores"].split(","))

df_base["n_autores"] = df_base.apply(contar_autores, axis=1)

# -----------------------------------------------------
# Citas fraccionales
# -----------------------------------------------------
df_base["citas_fraccionales"] = (
    df_base["citaciones"] / df_base["n_autores"]
)

# =====================================================
# 2. Calcular hm-index por autor (Schreiber)
# =====================================================
hm_dict = {}

for autor, grupo in df_base.groupby("autor"):
    citas_frac = (
        grupo["citas_fraccionales"]
        .sort_values(ascending=False)
        .values
    )

    hm = 0
    for i, c in enumerate(citas_frac, start=1):
        if c >= i:
            hm = i
        else:
            break

    hm_dict[autor] = hm

hm_df = (
    pd.DataFrame.from_dict(hm_dict, orient="index")
      .reset_index()
      .rename(columns={"index": "autor", 0: "hm_index"})
)

# =====================================================
# 3. Cruce con métricas de autoría ya calculadas
# =====================================================
df_full = resultado.merge(
    hm_df,
    on="autor",
    how="left"
)

df_full["hm_index"] = df_full["hm_index"].fillna(0)

# =====================================================
# 4. Métricas tipo Ioannidis
# =====================================================
df_full["NC"] = df_full["citas_totales"]
df_full["NCSF"] = df_full["citas_primer_autor"]
df_full["NCSFL"] = (
    df_full["citas_primer_autor"] + df_full["citas_ultimo_autor"]
)

# Citas en papers de autor único (aproximación estándar)
df_full["NCS"] = np.where(
    (df_full["primer_autor"] > 0) &
    (df_full["primer_autor"] == df_full["ultimo_autor"]),
    df_full["citas_primer_autor"],
    0
)

# =====================================================
# 5. Normalización (z-score)
# =====================================================
metricas_ci = [
    "NC",
    "NCS",
    "NCSF",
    "NCSFL",
    "hm_index"
]

for col in metricas_ci:
    df_full[f"z_{col}"] = zscore(
        df_full[col],
        nan_policy="omit"
    )

# =====================================================
# 6. Composite Index final (sin h-index)
# =====================================================
df_full["composite_index"] = (
    df_full["z_NC"] +
    df_full["z_NCS"] +
    df_full["z_NCSF"] +
    df_full["z_NCSFL"] +
    df_full["z_hm_index"]
)

# =====================================================
# 7. Ranking final
# =====================================================
df_full = df_full.sort_values(
    by="composite_index",
    ascending=False
)

df_full.head(10)


Loading ITables v2.6.2 from the internet... (need help?)


In [ ]:
display(df_full)

Loading ITables v2.6.2 from the internet... (need help?)
